In [3]:
import pulp
import time
from pulp import GUROBI
import haversine as hv
import pandas as pd


def haversine_dist(x, y):
    return hv.haversine(x, y, unit=hv.Unit.KILOMETERS)


sizes = [40,50,70]

#Parameter set
h = [0.5,0,0,0,0] #Heuristics
p = [-1,-1,-1,2,2] #Presolve
c = [-1,2,3,-1,3] #Cuts

#Create dictionary of decision variables and models
x = {}
t = {}
tsp = {}

#Heuristic solution routes from ORtools
# initial_values = [
#     [0, 6, 2, 9, 3, 4, 1, 8, 7, 5, 0],
#     [0, 18, 16, 17, 5, 12, 11, 7, 8, 13, 1, 4, 14, 19, 3, 9, 2, 15, 6, 10, 0],
#     [0, 18, 16, 17, 21, 10, 23, 6, 15, 2, 9, 3, 27, 19, 14, 4, 1, 13, 28, 8, 7, 29, 11, 12, 25, 26, 24, 5, 22, 20, 0],
#     [0, 20, 32, 22, 35, 18, 16, 17, 30, 5, 24, 26, 33, 39, 25, 12, 31, 11, 29, 7, 8, 28, 13, 1, 4, 14, 34, 19, 27, 3, 9, 38, 2, 15, 6, 23, 10, 21, 37, 36, 0],
#     [0, 10, 23, 6, 15, 2, 38, 9, 3, 44, 30, 27, 19, 34, 49, 14, 4, 1, 13, 45, 28, 8, 7, 42, 48, 5, 43, 29, 11, 31, 12, 25, 39, 33, 41, 46, 26, 24, 47, 40, 21, 37, 36, 17, 16, 18, 35, 22, 32, 20, 0],
#     [0, 18, 16, 17, 36, 10, 21, 37, 40, 59, 63, 58, 62, 66, 60, 67, 30, 54, 1, 4, 14, 49, 34, 19, 44, 38, 15, 6, 23, 2, 9, 3, 27, 65, 55, 13, 45, 28, 57, 8, 69, 50, 7, 42, 61, 48, 5, 43, 51, 29, 11, 31, 68, 56, 53, 12, 25, 39, 33, 41, 46, 26, 24, 47, 52, 22, 32, 20, 64, 35, 0],
#     [0, 35, 64, 20, 32, 95, 22, 96, 92, 58, 63, 62, 66, 52, 68, 31, 11, 56, 12, 53, 78, 25, 81, 74, 39, 89, 90, 33, 41, 46, 26, 24, 47, 97, 60, 70, 75, 30, 67, 86, 48, 5, 61, 43, 91, 87, 73, 85, 29, 51, 7, 93, 94, 42, 79, 71, 54, 1, 13, 84, 8, 83, 69, 50, 57, 28, 45, 72, 55, 82, 14, 4, 49, 34, 19, 65, 88, 27, 76, 44, 3, 9, 38, 2, 15, 6, 23, 40, 77, 99, 80, 59, 37, 21, 10, 98, 36, 17, 16, 18, 0]
# ]

#Create dataframe of results
result = pd.DataFrame(columns = ('No. of cities','Route Distance','Solution Time'))

#Create dataset, model and Solve for different No. of cities(sizes)
for param in range(len(h)):
    for size in sizes:
        coord = pd.read_csv('tsp_input.csv')[:size]
    #     coord.set_index('Place_Name', inplace=True)
        print(coord)
        dis_mat = [[haversine_dist((coord.loc[i]['Latitude'],coord.loc[i]['Longitude']), (coord.loc[j]['Latitude'],coord.loc[j]['Longitude'])) for j in coord.index] for i in coord.index]
        edges = [(i,j) for i in coord.index for j in coord.index]
        x[size] = {}

        #Create decision variable
        for e in edges:
            x[size][e] = pulp.LpVariable("x_" + str(size) + '_' + str(edges.index(e)), cat='Binary')
        tsp[size] = pulp.LpProblem("TSP_size"+str(size),pulp.LpMinimize)

        #Define objective function
        z = 0
        for e in edges:
            z += x[size][e]*dis_mat[e[0]][e[1]]
        tsp[size] += z

    #     #Warm start initial value
    #     for i in range(len(initial_values[sizes.index(size)][:-1])):
    #         x[size][initial_values[sizes.index(size)][i],initial_values[sizes.index(size)][i+1]].setInitialValue(1)


        t[size] = {}
        for i in coord.index:
            #Define Vertex sequence(subtour elimination) variable
            t[size][i] = pulp.LpVariable("t_" + str(size) + '_' + str(i),cat = 'Integer',lowBound = 1,upBound=size)
            #Define constraints
            tsp[size] += x[size][i,i]==0
            tsp[size] += pulp.lpSum(x[size][i,j] for j in coord.index) == 1
            tsp[size] += pulp.lpSum(x[size][j,i] for j in coord.index) == 1

        #Define subtour elimination constraint
        for i in coord.index:
            for j in coord.index:
                if(i!=j and list(coord.index).index(i)!=0 and list(coord.index).index(j)!=0):
                    tsp[size] += t[size][j]>=t[size][i] + 1 - 2*size*(1-x[size][i,j])
        tsp[size].writeLP('tsp_puLP_size' + str(size) + '.lp')

        #Solve
        solver_start_time = time.time()
        tsp[size].solve(GUROBI(Heuristics=h[param], cuts=c[param], Presolve=p[param]))
        solver_end_time = time.time()

        #Add Route distance and Solution time to results dataframe
        result.loc[len(result['No. of cities'])] = [size ,round(z.value(),2),round(solver_end_time - solver_start_time, 2)]
        print(result)
    result.to_csv('MIP_param%d_TSP_results.csv'%param)
    


                              Place_Name   Latitude  Longitude
0    Nanjangud, Mysore, Karnataka, India  12.120000  76.680000
1          Chittorgarh, Rajasthan, India  24.879999  74.629997
2          Ratnagiri, Maharashtra, India  16.994444  73.300003
3   Goregaon, Mumbai, Maharashtra, India  19.155001  72.849998
4             Pindwara, Rajasthan, India  24.794500  73.055000
5            Raipur, Chhattisgarh, India  21.250000  81.629997
6                Gokak, Karnataka, India  16.166700  74.833298
7          Lucknow, Uttar Pradesh, India  26.850000  80.949997
8                           Delhi, India  28.679079  77.069710
9             Mumbai, Maharashtra, India  19.076090  72.877426
10               Sagar, Karnataka, India  14.167040  75.040298
11        Jalpaiguri, West Bengal, India  26.540457  88.719391
12               Pakur, Jharkhand, India  24.633568  87.849251
13        Sardarshahar, Rajasthan, India  28.440554  74.493011
14              Sirohi, Rajasthan, India  24.882618  72